In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Phải đặt trước khi import torch

!pip install peft datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from peft import PrefixTuningConfig, TaskType, get_peft_model
import evaluate

MODEL_NAME = "t5-base"
CHECKPOINT_PATH = None  # Thay bằng đường dẫn checkpoint thực tế nếu có
OUTPUT_DIR = "/kaggle/working/prefix_qnli"
MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 10
NUM_PREFIX_TOKENS = 100
BATCH_SIZE = 32
LEARNING_RATE = 3e-4
NUM_EPOCHS = 10
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"Checkpoint: {CHECKPOINT_PATH if CHECKPOINT_PATH else 'None - training from scratch'}")

Device: cuda
  GPU 0: Tesla T4
Checkpoint: None - training from scratch


In [3]:
raw_dataset = load_dataset("glue", "qnli", split={"train": "train", "validation": "validation"})
print(raw_dataset)
print("\nSample train example:")
print(raw_dataset["train"][0])

README.md: 0.00B [00:00, ?B/s]

qnli/train-00000-of-00001.parquet:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

qnli/validation-00000-of-00001.parquet:   0%|          | 0.00/872k [00:00<?, ?B/s]

qnli/test-00000-of-00001.parquet:   0%|          | 0.00/877k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/104743 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5463 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5463 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 104743
    })
    validation: Dataset({
        features: ['question', 'sentence', 'label', 'idx'],
        num_rows: 5463
    })
})

Sample train example:
{'question': 'When did the third Digimon series begin?', 'sentence': 'Unlike the two seasons before it and most of the seasons that followed, Digimon Tamers takes a darker and more realistic approach to its story featuring Digimon who do not reincarnate after their deaths and more complex character development in the original Japanese.', 'label': 1, 'idx': 0}


In [4]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

LABEL_MAP = {0: "entailment", 1: "not_entailment"}
LABEL_REVERSE_MAP = {"entailment": 0, "not_entailment": 1}

def preprocess_function(examples):
    inputs = [
        f"qnli question: {q} sentence: {s}"
        for q, s in zip(examples["question"], examples["sentence"])
    ]
    targets = [LABEL_MAP[label] for label in examples["label"]]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Sửa lại vì raw_dataset giờ là dict thông thường, không phải DatasetDict
tokenized_dataset = {
    split: raw_dataset[split]
        .map(
            preprocess_function,
            batched=True,
            remove_columns=raw_dataset[split].column_names,
        )
    for split in ["train", "validation"]
}

print("Tokenized dataset:")
for split, ds in tokenized_dataset.items():
    print(f"  {split}: {len(ds)} examples")

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/104743 [00:00<?, ? examples/s]

Map:   0%|          | 0/5463 [00:00<?, ? examples/s]

Tokenized dataset:
  train: 104743 examples
  validation: 5463 examples


In [5]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

peft_config = PrefixTuningConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    num_virtual_tokens=NUM_PREFIX_TOKENS,
    prefix_projection=False,  # Soft prefix trực tiếp, đúng với paper
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

trainable params: 1,843,200 || all params: 224,746,752 || trainable%: 0.8201


In [6]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # Dùng startswith để xử lý trường hợp predict bị cắt ngắn
    def normalize(text):
        if text.startswith("not"):
            return 1  # not_entailment
        elif text.startswith("ent"):
            return 0  # entailment
        else:
            return -1  # không xác định được

    pred_ids = [normalize(p) for p in decoded_preds]
    label_ids = [normalize(l) for l in decoded_labels]

    valid = [(p, l) for p, l in zip(pred_ids, label_ids) if p != -1 and l != -1]
    if not valid:
        return {"accuracy": 0.0}

    valid_preds, valid_labels = zip(*valid)
    result = accuracy_metric.compute(predictions=valid_preds, references=valid_labels)
    return {"accuracy": round(result["accuracy"] * 100, 2)}

In [7]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",

    save_total_limit=3,

    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    warmup_steps=500,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

print("Training arguments configured successfully.")

Training arguments configured successfully.


In [8]:
model = model.to("cuda")

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

if CHECKPOINT_PATH:
    print(f"Resuming from checkpoint: {CHECKPOINT_PATH}")
    trainer.train(resume_from_checkpoint=CHECKPOINT_PATH)
else:
    print("Starting training from scratch...")
    trainer.train()

Starting training from scratch...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.033901,0.033863,93.260000
2,0.033178,0.033964,93.280000
3,0.033067,0.033875,93.300000
4,0.033214,0.033663,93.300000
5,0.033508,0.032296,93.230000


In [9]:
print("=" * 50)
print("Final Evaluation on Validation Set (GLUE-QNLI)")
print("=" * 50)

eval_results = trainer.evaluate(eval_dataset=tokenized_dataset["validation"])

print(f"\nAccuracy: {eval_results.get('eval_accuracy', 'N/A')}%")
print(f"Eval Loss: {eval_results.get('eval_loss', 'N/A'):.4f}")

print("\n--- So sánh với paper (Table 1, T5-base) ---")
print(f"Paper - Prefix Tuning (PF): 87.48%")
print(f"Paper - Fine-tuning  (FT):  92.57%")
print(f"Paper - LoRA         (LR):  92.02%")
print(f"Paper - Adapter      (AP):  91.58%")
print(f"Your model result:          {eval_results.get('eval_accuracy', 'N/A')}%")

Final Evaluation on Validation Set (GLUE-QNLI)



Accuracy: 93.3%
Eval Loss: 0.0339

--- So sánh với paper (Table 1, T5-base) ---
Paper - Prefix Tuning (PF): 87.48%
Paper - Fine-tuning  (FT):  92.57%
Paper - LoRA         (LR):  92.02%
Paper - Adapter      (AP):  91.58%
Your model result:          93.3%


In [10]:
# Lưu toàn bộ model
trainer.save_model(f"{OUTPUT_DIR}/final_model")

# Lưu riêng PEFT weights (chỉ phần prefix, rất nhỏ vài MB)
model.save_pretrained(f"{OUTPUT_DIR}/peft_weights")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/peft_weights")

print(f"Full model saved to:  {OUTPUT_DIR}/final_model")
print(f"PEFT weights saved to: {OUTPUT_DIR}/peft_weights")
print("\nĐể load lại model sau này:")
print("  from peft import PeftModel")
print(f"  base = T5ForConditionalGeneration.from_pretrained('{MODEL_NAME}')")
print(f"  model = PeftModel.from_pretrained(base, '{OUTPUT_DIR}/peft_weights')")

Full model saved to:  /kaggle/working/prefix_qnli/final_model
PEFT weights saved to: /kaggle/working/prefix_qnli/peft_weights

Để load lại model sau này:
  from peft import PeftModel
  base = T5ForConditionalGeneration.from_pretrained('t5-base')
  model = PeftModel.from_pretrained(base, '/kaggle/working/prefix_qnli/peft_weights')
